In [1]:
import json
import os
os.makedirs("data/json_files",exist_ok=True)

In [2]:
json_data={
  "companies": [
    {
      "name": "ABC Tech",
      "location": "New Delhi",
      "departments": [
        { "id": 1, "name": "Engineering" },
        { "id": 2, "name": "Design" }
      ],
      "employees": [
        {
          "id": 1,
          "name": "Rahul",
          "position": "Developer",
          "salary": 50000,
          "departmentId": 1
        },
        {
          "id": 2,
          "name": "Priya",
          "position": "Designer",
          "salary": 40000,
          "departmentId": 2
        }
      ]
    },
    {
      "name": "NextGen Solutions",
      "location": "Mumbai",
      "departments": [
        { "id": 3, "name": "Management" },
        { "id": 4, "name": "HR" }
      ],
      "employees": [
        {
          "id": 3,
          "name": "Amit",
          "position": "Manager",
          "salary": 70000,
          "departmentId": 3
        },
        {
          "id": 4,
          "name": "Sneha",
          "position": "HR Executive",
          "salary": 45000,
          "departmentId": 4
        }
      ]
    },
    {
      "name": "CodeWorks",
      "location": "Bangalore",
      "departments": [
        { "id": 5, "name": "Backend" },
        { "id": 6, "name": "Frontend" }
      ],
      "employees": [
        {
          "id": 5,
          "name": "Arjun",
          "position": "Backend Developer",
          "salary": 60000,
          "departmentId": 5
        },
        {
          "id": 6,
          "name": "Neha",
          "position": "Frontend Developer",
          "salary": 55000,
          "departmentId": 6
        }
      ]
    }
  ]
}

In [5]:
with open('data/json_files/company_data.json','w') as f:
    json.dump(json_data, f, indent=2)


In [6]:
jsonl_data=[
    {
      "timestamp": "2026-04-23T10:15:30Z",
      "event": "login",
      "user_id": 101,
      "page": "/home"
    },
    {
      "timestamp": "2026-04-23T10:20:45Z",
      "event": "view_page",
      "user_id": 102,
      "amount": 99.99
    },
    {
      "timestamp": "2026-04-23T10:25:10Z",
      "event": "logout",
      "user_id": 101
    }
  ]

with open('data/json_files/event_logs.jsonl','w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

In [15]:
# json processing libraries 

from langchain_community.document_loaders import JSONLoader
import json


# method 1 json loader with jq schema 
employee_loader= JSONLoader(
    file_path='data/json_files/company_data.json',
    jq_schema='.companies[].employees[]', #jq query to get each emplyee 
    text_content=False #full json objects    
)

employee_docs=employee_loader.load()
print(f"Loaded {len(employee_docs)} employee documents.")
print(f"First employee document content: {employee_docs[0].page_content}")

Loaded 6 employee documents.
First employee document content: {"id": 1, "name": "Rahul", "position": "Developer", "salary": 50000, "departmentId": 1}


In [16]:
from typing import List 
from langchain_core.documents import Document

print("custom json parsing")

def process_json_intelligentely(filepath: str)-> List[Document]:
    '''process json with intelligently flattening and context preservation '''
    with open(filepath,'r') as f:
        data=json.load(f)
    
    documents=[]
    #flattening nested structures
    for company in data.get('companies',[]):
        company_info={
            'name':company['name'],
            'location':company['location']
        }

        #process departments
        for dept in company.get('departments',[]):
            dept_info={
                **company_info,
                'department':dept['name']
            }
            #process employees
            for emp in company.get('employees',[]):
                if emp.get('departmentId')==dept['id']:
                    emp_info={
                        **dept_info,
                        'employee_name':emp['name'],
                        'position':emp['position'],
                        'salary':emp['salary']
                    }
                    documents.append(Document(
                        page_content=json.dumps(emp_info),
                        metadata={'source':filepath,'type':'employee'}
                    ))
    return documents

custom json parsing
